# 第14回　総括：孤高の意思決定者として
## ―― Ⅰ＝直感の敗北、Ⅱ＝集団の敗北。では、どう判断するのか

統計学Ⅱ　2026後期　／　北星学園大学　／　小野原 彩香

---

### このノートの使い方

最終回。新しい定理はもう出てこない。14回を1枚の地図にまとめ、全体を貫いていた **一本の糸** を、最後にもう一度、自分の目で確かめる。そして第1回の自分と再会する。▶ を上から押そう。

In [ ]:
!pip install -q japanize-matplotlib
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401
from scipy import stats
print("準備OK。次のセルへ。")

---
## 1. 14回の地図

```
確率論で武装する     02 条件付き確率（独立を定義）
                    03 ベイズ（信念の更新）  04 二項・ポアソン（独立試行）
                    05 正規分布（独立な和）
推測を問い直す       06 中心極限定理の再考   07 区間推定
誤用を見抜く         08 検定の多重性         09 効果量とp値批判
因果に近づく         10 RCTと観察研究        11 交絡と疑似相関
集団を解剖する       12 情報カスケード       13 コンドルセの陪審定理
統合                14 孤高の意思決定者として ← 今ここ
```

確率論 → 推測 → 因果 → 集団。バラバラに見える14回だが、実は **たった一つの概念** がすべてを貫いていた。

---
## 2. 一本の糸 ―― 独立性

今期ずっと言い続けてきた。

- 第4・5回：二項分布も正規分布も、**独立**な試行・要因から生まれる
- 第6・7回：中心極限定理も信頼区間も、標本が**独立**であることが前提
- 第10回：RCTは、ランダム化で処置を交絡から**独立**にして因果を取り出す
- 第12・13回：「空気を読む」とは判断の**独立**を手放すこと。集合知が崩れる

これが偶然の繰り返しでないことを、最後に1枚の図で確かめよう。**独立の度合い（ρ）を0から上げていく**と、性質のまったく違う2つの道具 ―― 第7回の **信頼区間** と第13回の **コンドルセの多数決** ―― が、**そろって同時に壊れていく**。

In [ ]:
rng = np.random.default_rng(14)

# (A) 95%信頼区間のカバレッジ（第7回）：独立が崩れると95%を割る
μ, σ, n_ci = 50, 10, 30
def CIカバレッジ(ρ, 試行=6000):
    c = 0
    for _ in range(試行):
        標本 = μ + σ * (np.sqrt(ρ) * rng.normal() + np.sqrt(1 - ρ) * rng.normal(size=n_ci))
        se = 標本.std(ddof=1) / np.sqrt(n_ci); t = stats.t.ppf(0.975, n_ci - 1); m = 標本.mean()
        c += (m - t*se <= μ <= m + t*se)
    return c / 試行

# (B) コンドルセの多数決（201人・一人55%）の正解率（第13回）：独立が崩れると崩壊
p = 0.55; τ = stats.norm.ppf(p)
def コンドルセ正解率(ρ, n=201, 試行=8000):
    C = rng.normal(size=(試行, 1)); E = rng.normal(size=(試行, n))
    X = np.sqrt(ρ) * C + np.sqrt(1 - ρ) * E
    return ((X < τ).sum(axis=1) > n / 2).mean()

ρリスト = [0.0, 0.1, 0.2, 0.3, 0.45, 0.6]
cov = [CIカバレッジ(ρ) for ρ in ρリスト]
cond = [コンドルセ正解率(ρ) for ρ in ρリスト]
for ρ, a, b in zip(ρリスト, cov, cond):
    print(f"独立が崩れる度 ρ={ρ}： 95%信頼区間のカバレッジ {a:.0%} ／ コンドルセ多数決の正解率 {b:.0%}")

In [ ]:
plt.figure(figsize=(7.5, 4.5))
plt.plot(ρリスト, cov, "o-", color="#3949ab", label="95%信頼区間のカバレッジ（第7回）")
plt.plot(ρリスト, cond, "s-", color="#e8503a", label="コンドルセ多数決の正解率（第13回）")
plt.axvline(0, color="gray", lw=0.8)
plt.xlabel("独立が崩れる度合い ρ（0＝完全に独立 → 右へ行くほど相関・空気を読む）")
plt.ylabel("道具が機能している度合い")
plt.title("独立という一本の糸：ρを上げると、別物の2つの道具がそろって壊れる")
plt.ylim(0, 1.02); plt.legend()
plt.show()
print("ρ=0（独立）では両方とも高い。独立が崩れた瞬間、まったく別の道具が同時に崩れ落ちる。")
print("→ これが偶然でない証拠。独立は、統計と集団の意思決定を支える共通の土台だった。")

**性質のまったく違う2つの道具が、$\rho$ を上げた瞬間、そろって崩れ落ちる。** これは偶然ではない。独立は、推測統計から集団の意思決定まで、すべてを支えていた共通の土台だったのだ。それが、この14回が伝えたかった一つのことだ。

---
## 3. 統計的誤用＋集団的誤り チェックリスト

これから君がデータや「みんなの意見」に出会ったとき、立ち止まって問うための11の質問。今期の全14回が、この1枚に凝縮されている。

1. 条件付き確率を **逆向き** に読んでいないか（P(A\|B) と P(B\|A) の混同）（→02）
2. **基準率** を無視していないか（→03）
3. 「**独立な試行**」という前提が本当に成り立っているか（→04・06）
4. **正規分布** を無条件に仮定していないか（→05）
5. 推測（CLT・区間・検定）の前提である **標本の独立性** が崩れていないか（→06・07）
6. 検定を **繰り返して** 偽陽性を増やしていないか（多重性・p-hacking）（→08）
7. p値を **効果の大きさ** と混同していないか／効果量を見たか（→09）
8. 観察データで安易に **因果** を主張していないか（→10）
9. **交絡** を疑ったか（見せかけの相関ではないか）（→11）
10. その「合意」は **情報カスケード** の産物ではないか（→12）
11. その多数決は **独立な判断の集積** か、それとも「空気を読んだ」結果か（→13）

統計学Ⅰのチェックリスト（相関≠因果・p値の誤解 など）と合わせれば、君はもう、たいていの統計的な嘘を見抜ける。

---
## 4. 第1回の自分と、再会する

第1回で、こう書いた ―― 「Ⅰでは“あなたの直感は外れる”を学んだ。Ⅱでは“集団になると、もっと外れる”を数学で証明する」。

証明は終わった。

- **Ⅰの結論**：人間の直感は、速いが系統的に外れる。
- **Ⅱの結論**：その人間が群れて空気を読むと、集団はさらに系統的に外れる。

では、どうすればいいのか。答えは悲観ではない。**道具を持て**、ということだ。

> 💬 **孤高の意思決定者として**
> 
> 統計は、確実な答えをくれる魔法ではない。不確実性は消えない。だが我々は ―― **ベイズで信念を更新し、効果量で大きさを測り、交絡を疑い、そして何より、独立した一個の人間として判断する** ことができる。群れず、空気に流されず、自分のシグナルを手放さない。それは孤独な作業かもしれない。けれど、コンドルセの定理が示したとおり、**その独立こそが、君を、そして君のいる集団を、賢くする**。これは精神論ではなく、この14回で君自身が数値で確かめた事実だ。

---
## 今日の課題（Moodle）と、これから

- **総合ふりかえり**：第1回に書いた「Ⅱへの期待」と今の自分を比べ、最も考えが変わったことを書く。
- **自由ミニ分析（任意）**：身の回りの「流行・世論・行列・多数決」を1つ選び、「これは独立な判断の集積か、空気の産物か」をチェックリストの言葉で論じる。

詳しくはMoodleの第14回課題を見ること。

---

半年間、おつかれさま。これからデータや「みんなの意見」に出会うたびに、今日の1枚の図を思い出してほしい ―― **独立は、君が思っているより、ずっと大切だ。**